### Analytics #1 — Top Scoring Batsmen per Season
-    SQL Query
-    Result Validation
-    Visualization

In [0]:
%sql
-- Get all batsman total runs in each season
WITH batsman_season_runs AS (
    SELECT
        season_clean AS season,
        batter,
        SUM(batsman_runs) AS total_runs
    FROM upskill.pyspark_learning.gold_delivery_match
    GROUP BY
        season_clean,
        batter
),

-- Ranked batsman in each season
ranked_batsmen AS (
    SELECT
        season,
        batter,
        total_runs,
        DENSE_RANK() OVER (
            PARTITION BY season
            ORDER BY total_runs DESC
        ) AS run_rank
    FROM batsman_season_runs
)

-- SELECT Top batsman in each season
SELECT
    season,
    batter,
    total_runs
FROM ranked_batsmen
WHERE run_rank = 1
ORDER BY
    season,
    run_rank,
    batter;

season,batter,total_runs
2008,SE Marsh,616
2009,ML Hayden,572
2010,SR Tendulkar,618
2011,CH Gayle,608
2012,CH Gayle,733
2013,MEK Hussey,733
2014,RV Uthappa,660
2015,DA Warner,562
2016,V Kohli,973
2017,DA Warner,641


Databricks visualization. Run in Databricks to view.

### Analytics #2 — Economical Bowlers — Powerplay
Which bowlers have the best economy rate during Powerplay overs across the entire dataset
-    SQL Query
-    Result Validation
-    Visualization

In [0]:
%sql
-- get total runs given by bowler during powerplay overs i.e first 6 overs
WITH powerplay_bowling AS
(
    SELECT
        bowler,

        SUM(
            batsman_runs
            + CASE
                WHEN extras_type IN ('wides', 'noballs')
                THEN extra_runs
                ELSE 0
              END
        ) AS runs_conceded,

        SUM(is_legal_delivery_flag) AS legal_deliveries

    FROM upskill.pyspark_learning.gold_delivery_match

    WHERE over < 6

    GROUP BY bowler
),

-- calculate economy rate for each bowler across all IPL seasons
bowler_economy AS
(
    SELECT
        bowler,
        runs_conceded,
        legal_deliveries,
        ROUND(
            runs_conceded / (legal_deliveries / 6.0),
            2
        ) AS economy

    FROM powerplay_bowling

    WHERE legal_deliveries >= 60
),

-- Rank bowler as per their economy rate
ranked_bowlers AS
(
    SELECT
        bowler,
        runs_conceded,
        legal_deliveries,
        economy,
        DENSE_RANK() OVER (
            ORDER BY economy
        ) AS economy_rank

    FROM bowler_economy
)

--  select bowlers within the top 10 economy ranks
SELECT
    bowler,
    runs_conceded,
    legal_deliveries,
    ROUND(legal_deliveries / 6.0, 1) AS overs,
    economy
FROM ranked_bowlers
WHERE economy_rank <= 10
ORDER BY economy;

bowler,runs_conceded,legal_deliveries,overs,economy
AG Murtaza,52,78,13.0,4.00
FH Edwards,58,78,13.0,4.46
A Kumble,88,108,18.0,4.89
SMSM Senanayake,110,126,21.0,5.24
A Symonds,106,114,19.0,5.58
GD McGrath,211,222,37.0,5.70
T Thushara,57,60,10.0,5.70
A Chandila,137,144,24.0,5.71
JC Archer,379,396,66.0,5.74
R Rampaul,161,162,27.0,5.96


Databricks visualization. Run in Databricks to view.

### Analytics #3 — Average Runs in Wins
When a team wins, what is its average score?
-    SQL Query
-    Result Validation
-    Visualization

In [0]:
%sql

-- calculate total runs scored by each winning team in each season
WITH winning_team_score AS
(
    SELECT
        match_id,
        season_clean AS season,
        winner,
        SUM(total_runs) AS winning_team_score

    FROM upskill.pyspark_learning.gold_delivery_match

    WHERE winner IS NOT NULL
      AND winner <> 'NA'
      AND match_outcome = 'Won'
      AND batting_team = winner

    GROUP BY
        match_id,
        season_clean,
        winner
),

-- Derive team name after standardization of team names
-- This is required as the team names are not consistent across the dataset
-- For example, below are the latest and old names.
-- (Rising Pune Supergiants(latest),Pune Warriors,Rising Pune Supergiant)
-- (Delhi Capitals(latest),Delhi Daredevils)
--(Royal Challengers Bengaluru(latest),Royal Challengers Bangalore)
--(Punjab Kings(latest),Kings XI Punjab)
--(Sunrisers Hyderabad(latest),Deccan Chargers)

standardized_team AS
(
    SELECT
        match_id,
        season,
        CASE
            WHEN winner IN (
                'Rising Pune Supergiants',
                'Pune Warriors',
                'Rising Pune Supergiant'
            ) THEN 'Rising Pune Supergiants'

            WHEN winner IN (
                'Delhi Capitals',
                'Delhi Daredevils'
            ) THEN 'Delhi Capitals'

            WHEN winner IN (
                'Royal Challengers Bengaluru',
                'Royal Challengers Bangalore'
            ) THEN 'Royal Challengers Bengaluru'

            WHEN winner IN (
                'Punjab Kings',
                'Kings XI Punjab'
            ) THEN 'Punjab Kings'

            WHEN winner IN (
                'Sunrisers Hyderabad',
                'Deccan Chargers'
            ) THEN 'Sunrisers Hyderabad'

            ELSE winner
        END AS team,

        winning_team_score

    FROM winning_team_score
)

SELECT
    team,
    COUNT(*) AS wins,
    ROUND(AVG(winning_team_score), 2) AS avg_winning_team_score
FROM standardized_team
GROUP BY team
ORDER BY avg_winning_team_score DESC;

team,wins,avg_winning_team_score
Lucknow Super Giants,24,183.88
Gujarat Titans,28,175.57
Chennai Super Kings,138,169.11
Rajasthan Royals,110,168.63
Mumbai Indians,142,167.97
Punjab Kings,109,167.77
Royal Challengers Bengaluru,121,167.48
Gujarat Lions,13,167.08
Delhi Capitals,112,166.15
Sunrisers Hyderabad,116,163.5


Databricks visualization. Run in Databricks to view.

### Analytics #4 — Scores by Venue
Which venues tend to produce higher-scoring matches?
-    SQL Query
-    Result Validation
-    Visualization

In [0]:
%sql

-- calculate total runs scored for each match played in each venue 
WITH match_total_score AS
(
    SELECT
        match_id,
        venue,
        SUM(total_runs) AS match_total_runs
    FROM upskill.pyspark_learning.gold_delivery_match
    GROUP BY
        match_id,
        venue
),

-- Dervie matches played and average match score for each venue
venue_scores AS
(
    SELECT
        venue,
        COUNT(*) AS matches_played,
        ROUND(AVG(match_total_runs), 2) AS avg_match_score
    FROM match_total_score
    GROUP BY venue
)

-- Getvenues with average match scores where at least 10 matches were played
SELECT
    venue,
    matches_played,
    avg_match_score
FROM venue_scores
WHERE matches_played >= 10 --use at least 10 matches to make the comparison more reliable
ORDER BY avg_match_score DESC;

venue,matches_played,avg_match_score
"Arun Jaitley Stadium, Delhi",16,380.69
"Eden Gardens, Kolkata",16,380.31
"M Chinnaswamy Stadium, Bengaluru",14,380.14
"Rajiv Gandhi International Stadium, Uppal, Hyderabad",13,364.85
Brabourne Stadium,10,348.1
Punjab Cricket Association IS Bindra Stadium,10,347.6
"Wankhede Stadium, Mumbai",45,346.38
"Brabourne Stadium, Mumbai",17,342.59
"Sawai Mansingh Stadium, Jaipur",10,342.1
M.Chinnaswamy Stadium,15,341.8


Databricks visualization. Run in Databricks to view.

### Analytics #5 — Dismissal-kind analysis
How are batsmen most commonly dismissed in IPL matches?
-    SQL Query
-    Result Validation
-    Visualization

In [0]:
%sql

WITH dismissal_counts AS
(
    SELECT
        dismissal_kind,
        COUNT(*) AS dismissals
    FROM upskill.pyspark_learning.gold_delivery_match
    WHERE dismissal_kind IS NOT NULL
    GROUP BY dismissal_kind
)

SELECT
    dismissal_kind,
    dismissals,
    ROUND(
        100.0 * dismissals / SUM(dismissals) OVER (),
        2
    ) AS dismissal_pct
FROM dismissal_counts
ORDER BY dismissals DESC;

dismissal_kind,dismissals,dismissal_pct
caught,8063,62.26
bowled,2212,17.08
run out,1114,8.60
lbw,800,6.18
caught and bowled,367,2.83
stumped,358,2.76
hit wicket,15,0.12
retired hurt,15,0.12
obstructing the field,3,0.02
retired out,3,0.02


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    is_legal_delivery,
    is_legal_delivery_flag,
    COUNT(*) AS delivery_count
FROM upskill.pyspark_learning.gold_delivery_match
GROUP BY
    is_legal_delivery,
    is_legal_delivery_flag
ORDER BY
    is_legal_delivery,
    is_legal_delivery_flag;

is_legal_delivery,is_legal_delivery_flag,delivery_count
false,0,9449
true,1,251471


In [0]:
%sql

SELECT COUNT(*) AS total_matches
FROM upskill.pyspark_learning.silver_matches;

total_matches
1095


In [0]:
%sql

SELECT
    COUNT(DISTINCT season_clean) AS total_seasons
FROM upskill.pyspark_learning.silver_matches;

total_seasons
17
